# Noise model 구성하기

## 양자 회로에서 자주 다루는 error 종류

Noise model은 실제 양자 장치에서 발생하는 불완전성을 확률적 또는 물리적 channel로 표현합니다. 대표적인 error는 다음과 같습니다.

- **Bit-flip error ($X$ error)**: $|0\rangle$과 $|1\rangle$이 서로 뒤바뀌는 error입니다.
- **Phase-flip error ($Z$ error)**: 상태의 상대 위상 부호가 바뀌는 error입니다.
- **Bit-phase-flip error ($Y$ error)**: bit-flip과 phase-flip이 함께 발생하는 error입니다.
- **Pauli error**: $I$, $X$, $Y$, $Z$ 연산이 지정된 확률로 발생하는 일반적인 error channel입니다.
- **Depolarizing error**: 일정 확률로 qubit 상태가 무작위화되어 maximally mixed state 쪽으로 이동하는 error입니다.
- **Amplitude damping**: 에너지 손실로 excited state $|1\rangle$이 ground state $|0\rangle$으로 이완되는 현상입니다. 주로 $T_1$ relaxation과 관련됩니다.
- **Phase damping / Dephasing**: 에너지 변화 없이 상대 위상 정보가 사라지는 현상입니다. 주로 $T_2$ coherence time과 관련됩니다.
- **Thermal relaxation error**: $T_1$, $T_2$, gate time, thermal population을 이용해 relaxation과 dephasing을 함께 모델링합니다.
- **Reset error**: reset 후 qubit이 의도한 $|0\rangle$ 대신 $|1\rangle$ 등에 남는 error입니다.
- **Readout error (measurement error)**: 실제 측정값과 기록된 classical bit가 다르게 배정되는 error입니다.
- **Coherent unitary error**: gate angle의 과회전·저회전처럼 동일한 방향으로 누적될 수 있는 체계적인 unitary error입니다.
- **Mixed unitary error**: 여러 unitary 연산 중 하나가 각 확률에 따라 적용되는 error입니다.
- **Kraus error**: Kraus operator를 이용해 표현하는 일반적인 CPTP quantum channel입니다.
- **Crosstalk / non-local error**: 한 qubit의 연산이 주변 qubit에 영향을 주는 상관된 error입니다.

이 notebook에서는 특히 `Pauli error`, `depolarizing_error`, `ReadoutError`, `thermal_relaxation_error`와 이들을 `NoiseModel`에 추가하는 방법을 다룹니다.



{/*
  이 CELL을 수정하지 마십시오!!!
  이 cell의 내용은 script에 의해 자동 생성됩니다. 여기에 추가한 내용은
  notebook을 다음에 실행할 때 제거됩니다. 새 내용을 추가하려면 이 cell의
  앞이나 뒤에 새 cell을 만드십시오.
  */}

<Accordion>
  <AccordionItem title="Package version">
    이 페이지의 코드는 다음 requirements를 사용하여 개발되었습니다.
    아래 version 또는 그 이후 version 사용을 권장합니다.

    ```
    qiskit[all]~=2.5.2
    qiskit-ibm-runtime~=0.47.0
    qiskit-aer~=0.17
    ```
  </AccordionItem>
</Accordion>



이 페이지에서는 Qiskit Aer의 [`noise`](https://qiskit.org/ecosystem/aer/apidocs/aer_noise.html) module을 사용하여 error가 존재하는 quantum circuit을 simulation하기 위한 noise model을 구성하는 방법을 설명합니다. 이는 noisy quantum processor를 모사하고 noise가 quantum algorithm 실행에 미치는 영향을 연구할 때 유용합니다.



In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator

# Qiskit Aer noise module에서 import
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)

## Qiskit Aer `noise` module

Qiskit Aer의 `noise` module에는 simulation용 사용자 정의 noise model을 구성하는 Python class가 포함되어 있습니다. 주요 class는 세 가지입니다.

1. Noisy simulation에 사용할 noise model을 저장하는 `NoiseModel` class

2. CPTP gate error를 기술하는 `QuantumError` class. 다음 위치에 적용할 수 있습니다.
   * *gate* 또는 *reset* instruction 뒤
   * *measure* instruction 앞

3. Classical readout error를 기술하는 `ReadoutError` class



## Backend에서 noise model 초기화하기

Physical backend의 최신 calibration data로 설정된 parameter를 사용하여 noise model을 초기화할 수 있습니다.

<Admonition type="note">
  이 예제에서는 `qiskit_ibm_runtime`의 `FakeSherbrooke` mock backend를 사용하지만, Qiskit과 호환되는 어떤 real 또는 fake backend에서도 시도할 수 있습니다.
</Admonition>



In [2]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
noise_model = NoiseModel.from_backend(backend)

이 코드는 해당 backend를 사용할 때 발생할 error를 대략적으로 근사한 noise model을 생성합니다. Noise model의 parameter를 더 세밀하게 제어하려면 이 페이지의 나머지 부분에서 설명하는 것처럼 직접 noise model을 만들어야 합니다.



## Quantum error

`QuantumError` object를 직접 다루는 대신, parameter가 지정된 특정 종류의 quantum error를 자동으로 생성하는 여러 helper function을 사용할 수 있습니다. 이러한 함수는 `noise` module에 포함되어 있으며, quantum computing 연구에서 흔히 사용하는 다양한 error type을 지원합니다. Function 이름과 반환하는 error type은 다음과 같습니다.

| Standard error function         | 설명 |
| ------------------------------- | ---- |
| `kraus_error`                   | Kraus matrix 목록 $[K_0, ...]$으로 주어지는 일반적인 n-qubit CPTP error channel |
| `mixed_unitary_error`           | Unitary matrix와 확률의 목록 $[(U_0, p_0),...]$으로 주어지는 n-qubit mixed unitary error |
| `coherent_unitary_error`        | 하나의 unitary matrix $U$로 주어지는 n-qubit coherent unitary error |
| `pauli_error`                   | Pauli와 확률의 목록 $[(P_0, p_0),...]$으로 주어지는 n-qubit Pauli error channel (mixed unitary) |
| `depolarizing_error`            | Depolarization probability $p$로 parameter화된 n-qubit depolarizing error channel |
| `reset_error`                   | $\vert0\rangle$, $\vert1\rangle$ 상태로 reset될 확률 $p_0$, $p_1$로 parameter화된 single-qubit reset error |
| `thermal_relaxation_error`      | Relaxation time constant $T_1$, $T_2$, gate time $t$, excited-state thermal population $p_1$로 parameter화된 single-qubit thermal relaxation channel |
| `phase_amplitude_damping_error` | Amplitude damping parameter $\lambda$, phase damping parameter $\gamma$, excited-state thermal population $p_1$으로 주어지는 single-qubit generalized phase-amplitude damping error channel |
| `amplitude_damping_error`       | Amplitude damping parameter $\lambda$와 excited-state thermal population $p_1$으로 주어지는 single-qubit generalized amplitude damping error channel |
| `phase_damping_error`           | Phase damping parameter $\gamma$로 주어지는 single-qubit phase damping error channel |

### Quantum error 결합하기

`QuantumError` instance는 composition, tensor product, tensor expansion(순서가 반대인 tensor product)을 사용하여 다음과 같이 새로운 `QuantumError`로 결합할 수 있습니다.

* Composition: $\cal{E}(\rho)=\cal{E_2}(\cal{E_1}(\rho))$ → `error = error1.compose(error2)`
* Tensor product: $\cal{E}(\rho) =(\cal{E_1}\otimes\cal{E_2})(\rho)$ → `error = error1.tensor(error2)`
* Expand product: $\cal{E}(\rho) =(\cal{E_2}\otimes\cal{E_1})(\rho)$ → `error = error1.expand(error2)`



### 예제

5% 확률의 single-qubit bit-flip error를 구성하려면 다음과 같이 합니다.



In [3]:
# 1-qubit bit-flip error와 phase-flip error 구성
p_error = 0.05
bit_flip = pauli_error([("X", p_error), ("I", 1 - p_error)])
phase_flip = pauli_error([("Z", p_error), ("I", 1 - p_error)])
print(bit_flip)
print(phase_flip)

QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.05, Circuit = 
   ┌───┐
q: ┤ X ├
   └───┘
  P(1) = 0.95, Circuit = 
   ┌───┐
q: ┤ I ├
   └───┘
QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.05, Circuit = 
   ┌───┐
q: ┤ Z ├
   └───┘
  P(1) = 0.95, Circuit = 
   ┌───┐
q: ┤ I ├
   └───┘


In [4]:
# Bit-flip error와 phase-flip error를 composition으로 결합
bitphase_flip = bit_flip.compose(phase_flip)
print(bitphase_flip)

QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.0025000000000000005, Circuit = 
   ┌───┐┌───┐
q: ┤ X ├┤ Z ├
   └───┘└───┘
  P(1) = 0.0475, Circuit = 
   ┌───┐┌───┐
q: ┤ X ├┤ I ├
   └───┘└───┘
  P(2) = 0.0475, Circuit = 
   ┌───┐┌───┐
q: ┤ I ├┤ Z ├
   └───┘└───┘
  P(3) = 0.9025, Circuit = 
   ┌───┐┌───┐
q: ┤ I ├┤ I ├
   └───┘└───┘


In [5]:
# Bit-flip error와 phase-flip error의 tensor product 구성
# qubit-0에는 bit-flip, qubit-1에는 phase-flip 적용
error2 = phase_flip.tensor(bit_flip)
print(error2)

QuantumError on 2 qubits. Noise circuits:
  P(0) = 0.0025000000000000005, Circuit = 
     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ Z ├
     └───┘
  P(1) = 0.0475, Circuit = 
     ┌───┐
q_0: ┤ I ├
     ├───┤
q_1: ┤ Z ├
     └───┘
  P(2) = 0.0475, Circuit = 
     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ I ├
     └───┘
  P(3) = 0.9025, Circuit = 
     ┌───┐
q_0: ┤ I ├
     ├───┤
q_1: ┤ I ├
     └───┘


### QuantumChannel operator와 상호 변환하기

Qiskit Aer의 `QuantumError` object와 Qiskit의 `QuantumChannel` object 사이를 양방향으로 변환할 수도 있습니다.



In [6]:
# Kraus operator로 변환
bit_flip_kraus = Kraus(bit_flip)
print(bit_flip_kraus)

Kraus([[[-9.74679434e-01+1.19363805e-16j,
          0.00000000e+00+0.00000000e+00j],
        [ 0.00000000e+00+0.00000000e+00j,
         -9.74679434e-01+0.00000000e+00j]],

       [[ 0.00000000e+00+0.00000000e+00j,
          4.69834045e-18-2.23606798e-01j],
        [-2.26855945e-17-2.23606798e-01j,
          5.03721386e-33+4.96506831e-17j]]],
      input_dims=(2,), output_dims=(2,))


In [7]:
# Superoperator로 변환
phase_flip_sop = SuperOp(phase_flip)
print(phase_flip_sop)

SuperOp([[1. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
         [0. +0.j, 0.9+0.j, 0. +0.j, 0. +0.j],
         [0. +0.j, 0. +0.j, 0.9+0.j, 0. +0.j],
         [0. +0.j, 0. +0.j, 0. +0.j, 1. +0.j]],
        input_dims=(2,), output_dims=(2,))


In [8]:
# 다시 quantum error로 변환
print(QuantumError(bit_flip_kraus))

# 변환 결과가 원래 error와 동등한지 확인
QuantumError(bit_flip_kraus) == bit_flip

QuantumError on 1 qubits. Noise circuits:
  P(0) = 1.0, Circuit = 
   ┌───────┐
q: ┤ kraus ├
   └───────┘


True

### Readout error

Classical readout error는 assignment probability vector $P(A|B)$의 목록으로 지정합니다.

* $A$는 *기록된(recorded)* classical bit 값입니다.
* $B$는 측정에서 반환된 *실제(true)* bit 값입니다.

예를 들어 single qubit에서는 $P(A|B) = [P(A|0), P(A|1)]$입니다.



In [9]:
# Measurement 결과가 잘못 배정될 확률
p0given1 = 0.1
p1given0 = 0.05

ReadoutError([[1 - p1given0, p1given0], [p0given1, 1 - p0given1]])

ReadoutError([[0.95 0.05]
 [0.1  0.9 ]])

Readout error도 quantum error와 마찬가지로 `compose`, `tensor`, `expand`를 사용하여 결합할 수 있습니다.



## Noise model에 error 추가하기

Noise model에 quantum error를 추가할 때는 해당 error가 작용할 *instruction*의 종류와 적용할 qubit을 지정해야 합니다. Quantum error에는 두 가지 경우가 있습니다.

1. 모든 qubit에 적용되는 quantum error
2. 특정 qubit에 적용되는 quantum error

### 1. 모든 qubit에 적용되는 quantum error

Instruction이 어느 qubit에 작용하는지와 관계없이 해당 instruction이 나타날 때마다 같은 error를 적용합니다.

`noise_model.add_all_qubit_quantum_error(error, instructions)`로 추가합니다.



In [10]:
# 빈 noise model 생성
noise_model = NoiseModel()

# 모든 single-qubit u1, u2, u3 gate에 depolarizing error 추가
error = depolarizing_error(0.05, 1)
noise_model.add_all_qubit_quantum_error(error, ["u1", "u2", "u3"])

# Noise model 정보 출력
print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['u2', 'u3', 'u1']
  All-qubits errors: ['u1', 'u2', 'u3']


### 2. 특정 qubit에 적용되는 quantum error

지정한 qubit 목록에 작용하는 instruction이 나타날 때마다 error를 적용합니다. Qubit의 순서가 중요하다는 점에 유의하십시오. 예를 들어 two-qubit gate에서 qubit \[0, 1]에 적용되는 error와 qubit \[1, 0]에 적용되는 error는 서로 다릅니다.

`noise_model.add_quantum_error(error, instructions, qubits)`로 추가합니다.



In [11]:
# 빈 noise model 생성
noise_model = NoiseModel()

# Qubit 0의 모든 single-qubit u1, u2, u3 gate에만 depolarizing error 추가
error = depolarizing_error(0.05, 1)
noise_model.add_quantum_error(error, ["u1", "u2", "u3"], [0])

# Noise model 정보 출력
print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['u2', 'u3', 'u1']
  Qubits with noise: [0]
  Specific qubit errors: [('u1', (0,)), ('u2', (0,)), ('u3', (0,))]


### Non-local qubit quantum error에 관한 참고 사항

`NoiseModel`은 non-local qubit quantum error 추가를 지원하지 않습니다. 이러한 error는 `NoiseModel` 외부에서 처리해야 합니다. 자체 조건에 따라 circuit에 quantum error를 삽입해야 한다면 [사용자 정의 transpiler pass 작성](/docs/guides/custom-transpiler-pass)(`TransformationPass`) 후 simulator 실행 직전에 해당 pass를 실행하는 방법을 사용할 수 있습니다.



### Noise model로 noisy simulation 실행하기

`AerSimulator(noise_model=noise_model)` 명령은 주어진 noise model에 맞게 구성된 simulator를 반환합니다. Simulator의 noise model을 설정할 뿐 아니라 noise model의 gate에 맞추어 simulator의 basis gate도 재정의합니다.



## Noise model 예제

이제 몇 가지 noise model 예제를 살펴봅니다. 예제에서는 n-qubit GHZ state를 생성하는 간단한 test circuit을 사용합니다.



In [12]:
# System 설정
n_qubits = 4
circ = QuantumCircuit(n_qubits)

# Test circuit
circ.h(0)
for qubit in range(n_qubits - 1):
    circ.cx(qubit, qubit + 1)
circ.measure_all()
print(circ)

        ┌───┐                ░ ┌─┐         
   q_0: ┤ H ├──■─────────────░─┤M├─────────
        └───┘┌─┴─┐           ░ └╥┘┌─┐      
   q_1: ─────┤ X ├──■────────░──╫─┤M├──────
             └───┘┌─┴─┐      ░  ║ └╥┘┌─┐   
   q_2: ──────────┤ X ├──■───░──╫──╫─┤M├───
                  └───┘┌─┴─┐ ░  ║  ║ └╥┘┌─┐
   q_3: ───────────────┤ X ├─░──╫──╫──╫─┤M├
                       └───┘ ░  ║  ║  ║ └╥┘
meas: 4/════════════════════════╩══╩══╩══╩═
                                0  1  2  3 


### Ideal simulation

Noise가 없는 이상적인 결과를 먼저 확인합니다.



In [13]:
# Ideal simulator 구성 및 실행
sim_ideal = AerSimulator()
result_ideal = sim_ideal.run(circ).result()
plot_histogram(result_ideal.get_counts(0))

<Image src="/docs/images/guides/build-noise-models/extracted-outputs/bc713d11-755e-41e4-94f0-1ed76e3c2469-0.svg" alt="Output of the previous code cell" />

## Noise 예제 1: 기본 bit-flip error noise model

Quantum information theory 연구에서 흔히 사용하는 간단한 toy noise model을 살펴봅니다.

* Single-qubit gate를 적용할 때 `p_gate1`의 확률로 qubit 상태를 flip합니다.
* Two-qubit gate를 적용할 때 각 qubit에 single-qubit error를 적용합니다.
* Qubit을 reset할 때 `p_reset`의 확률로 0 대신 1로 reset합니다.
* Qubit을 측정할 때 `p_meas`의 확률로 qubit 상태를 flip합니다.



In [14]:
# 예제용 error probability
p_reset = 0.03
p_meas = 0.1
p_gate1 = 0.05

# QuantumError object
error_reset = pauli_error([("X", p_reset), ("I", 1 - p_reset)])
error_meas = pauli_error([("X", p_meas), ("I", 1 - p_meas)])
error_gate1 = pauli_error([("X", p_gate1), ("I", 1 - p_gate1)])
error_gate2 = error_gate1.tensor(error_gate1)

# Noise model에 error 추가
noise_bit_flip = NoiseModel()
noise_bit_flip.add_all_qubit_quantum_error(error_reset, "reset")
noise_bit_flip.add_all_qubit_quantum_error(error_meas, "measure")
noise_bit_flip.add_all_qubit_quantum_error(error_gate1, ["u1", "u2", "u3"])
noise_bit_flip.add_all_qubit_quantum_error(error_gate2, ["cx"])

print(noise_bit_flip)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['u2', 'u1', 'reset', 'cx', 'measure', 'u3']
  All-qubits errors: ['reset', 'measure', 'u1', 'u2', 'u3', 'cx']


### Noisy simulation 실행하기



In [15]:
# Noisy simulator backend 생성
sim_noise = AerSimulator(noise_model=noise_bit_flip)

# Noisy basis gate에 맞게 circuit transpile
passmanager = generate_preset_pass_manager(
    optimization_level=3, backend=sim_noise
)
circ_tnoise = passmanager.run(circ)

# 실행 후 count 가져오기
result_bit_flip = sim_noise.run(circ_tnoise).result()
counts_bit_flip = result_bit_flip.get_counts(0)

# Noisy output 시각화
plot_histogram(counts_bit_flip)

<Image src="/docs/images/guides/build-noise-models/extracted-outputs/abeb9f09-d762-406d-983e-0357ade59636-0.svg" alt="Output of the previous code cell" />

## 예제 2: T1/T2 thermal relaxation

이제 qubit과 환경 사이의 thermal relaxation을 기반으로 한, 더 현실적인 error model을 살펴봅니다.

* 각 qubit은 thermal relaxation time constant $T_1$과 dephasing time constant $T_2$로 parameter화됩니다.
* 반드시 $T_2 \le 2 T_1$을 만족해야 합니다.
* Instruction의 error rate는 gate time과 qubit의 $T_1$, $T_2$ 값으로 결정됩니다.



In [16]:
# Qubit 0-3의 T1 및 T2 값
T1s = np.random.normal(
    50e3, 10e3, 4
)  # 평균 50 microsecond인 normal distribution에서 sampling
T2s = np.random.normal(
    70e3, 10e3, 4
)  # 평균 70 microsecond인 normal distribution에서 sampling

# 무작위 T2가 T2 <= 2*T1 조건을 만족하도록 제한
T2s = np.array([min(T2s[j], 2 * T1s[j]) for j in range(4)])

# Instruction time (nanosecond 단위)
time_u1 = 0  # virtual gate
time_u2 = 50  # single X90 pulse
time_u3 = 100  # two X90 pulse
time_cx = 300
time_reset = 1000  # 1 microsecond
time_measure = 1000  # 1 microsecond

# QuantumError object
errors_reset = [
    thermal_relaxation_error(t1, t2, time_reset) for t1, t2 in zip(T1s, T2s)
]
errors_measure = [
    thermal_relaxation_error(t1, t2, time_measure) for t1, t2 in zip(T1s, T2s)
]
errors_u1 = [
    thermal_relaxation_error(t1, t2, time_u1) for t1, t2 in zip(T1s, T2s)
]
errors_u2 = [
    thermal_relaxation_error(t1, t2, time_u2) for t1, t2 in zip(T1s, T2s)
]
errors_u3 = [
    thermal_relaxation_error(t1, t2, time_u3) for t1, t2 in zip(T1s, T2s)
]
errors_cx = [
    [
        thermal_relaxation_error(t1a, t2a, time_cx).expand(
            thermal_relaxation_error(t1b, t2b, time_cx)
        )
        for t1a, t2a in zip(T1s, T2s)
    ]
    for t1b, t2b in zip(T1s, T2s)
]

# Noise model에 error 추가
noise_thermal = NoiseModel()
for j in range(4):
    noise_thermal.add_quantum_error(errors_reset[j], "reset", [j])
    noise_thermal.add_quantum_error(errors_measure[j], "measure", [j])
    noise_thermal.add_quantum_error(errors_u1[j], "u1", [j])
    noise_thermal.add_quantum_error(errors_u2[j], "u2", [j])
    noise_thermal.add_quantum_error(errors_u3[j], "u3", [j])
    for k in range(4):
        noise_thermal.add_quantum_error(errors_cx[j][k], "cx", [j, k])

print(noise_thermal)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u2', 'u3']
  Instructions with noise: ['u2', 'reset', 'cx', 'measure', 'u3']
  Qubits with noise: [0, 1, 2, 3]
  Specific qubit errors: [('reset', (0,)), ('reset', (1,)), ('reset', (2,)), ('reset', (3,)), ('measure', (0,)), ('measure', (1,)), ('measure', (2,)), ('measure', (3,)), ('u2', (0,)), ('u2', (1,)), ('u2', (2,)), ('u2', (3,)), ('u3', (0,)), ('u3', (1,)), ('u3', (2,)), ('u3', (3,)), ('cx', (0, 0)), ('cx', (0, 1)), ('cx', (0, 2)), ('cx', (0, 3)), ('cx', (1, 0)), ('cx', (1, 1)), ('cx', (1, 2)), ('cx', (1, 3)), ('cx', (2, 0)), ('cx', (2, 1)), ('cx', (2, 2)), ('cx', (2, 3)), ('cx', (3, 0)), ('cx', (3, 1)), ('cx', (3, 2)), ('cx', (3, 3))]


### Noisy simulation 실행하기



In [17]:
# Noisy simulation 실행
sim_thermal = AerSimulator(noise_model=noise_thermal)

# Noisy basis gate에 맞게 circuit transpile
passmanager = generate_preset_pass_manager(
    optimization_level=3, backend=sim_thermal
)
circ_tthermal = passmanager.run(circ)

# 실행 후 count 가져오기
result_thermal = sim_thermal.run(circ_tthermal).result()
counts_thermal = result_thermal.get_counts(0)

# Noisy output 시각화
plot_histogram(counts_thermal)

<Image src="/docs/images/guides/build-noise-models/extracted-outputs/ff52bf52-1323-40fc-a631-2b1889b21b20-0.svg" alt="Output of the previous code cell" />

## 다음 단계

<Admonition type="tip" title="권장 자료">
  * Noisy circuit을 simulation하려면 [Qiskit Aer primitive를 이용한 exact 및 noisy simulation](/docs/guides/simulate-with-qiskit-sdk-primitives)을 참고하십시오.
  * [Qiskit Aer noise module](https://qiskit.org/ecosystem/aer/apidocs/aer_noise.html) reference를 확인하십시오.
</Admonition>



© IBM Corp., 2017-2026